In [3]:
import glob
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from scipy.interpolate import UnivariateSpline

# ==========================================
# 1. HELPER FUNCTIE VOOR INLEZEN CSV-BESTANDEN
# ==========================================
def load_lab_csv(filepath):
    """Leest laboratorium CSV-bestanden in door kopregels over te slaan."""
    with open(filepath, 'r') as f:
        lines = f.readlines()
    
    data_start = 0
    for i, line in enumerate(lines):
        line_str = line.strip()
        if line_str and not '\t' in line_str:
            try:
                float(line_str.split(',')[0])
                data_start = i
                break
            except ValueError:
                pass
                
    values = [float(l.strip()) for l in lines[data_start:] if l.strip()]
    return np.array(values)

# ==========================================
# 2. PT1000 KALIBRATIE
# ==========================================
R_pt1000_77K = 201.7     # Gemeten Pt1000 weerstand bij 77 K (Ohm)[cite: 16]
R_pt1000_294K = 1088.0   # Gemeten Pt1000 weerstand bij 294 K (Ohm)[cite: 16]

a_cal = (294.0 - 77.0) / (R_pt1000_294K - R_pt1000_77K)
b_cal = 77.0 - a_cal * R_pt1000_77K

def pt1000_to_kelvin(R_sensor):
    return a_cal * R_sensor + b_cal

# ==========================================
# 3. DATA INLEZEN UIT JOUW CSV BESTANDEN
# ==========================================
# Pas onderstaande bestandsnamen aan naar de namen van jouw bestanden:
dmm_file = 'DMMdata_pt1000.csv'        # CSV met Pt1000 sensor data[cite: 13, 16]
smu_pos_file = 'SMUdata_pos_01A.csv'   # CSV met spanning bij +0.1 A[cite: 16, 18]
smu_neg_file = 'SMUdata_neg_01A.csv'   # CSV met spanning bij -0.1 A[cite: 16, 18]

# Hier worden de variabelen ingelezen die de foutmelding veroorzaakten:
R_pt1000_raw = load_lab_csv(dmm_file)
V_pos_raw = load_lab_csv(smu_pos_file)
V_neg_raw = load_lab_csv(smu_neg_file)

# ==========================================
# 4. VERWERKING & Tc BEPALING
# ==========================================
# Bereken Temperatuur
T_data = pt1000_to_kelvin(R_pt1000_raw)

# Bereken Gecorrigeerde BSCCO Weerstand (Offset Compensatie)[cite: 18, 19]
I_pos = 0.1
I_neg = -0.1
R_bscco_corrected = (V_pos_raw - V_neg_raw) / (I_pos - I_neg)

# Sorteren op stijgende temperatuur
sort_idx = np.argsort(T_data)
T_clean = T_data[sort_idx]
R_clean = R_bscco_corrected[sort_idx]

# Ruisfiltering & Spline Fit
window_len = 15 if len(R_clean) >= 15 else (len(R_clean) if len(R_clean) % 2 != 0 else len(R_clean) - 1)
R_smooth = savgol_filter(R_clean, window_length=window_len, polyorder=2)

spline = UnivariateSpline(T_clean, R_smooth, s=1e-6)
T_fine = np.linspace(T_clean.min(), T_clean.max(), 3000)
R_fit = spline(T_fine)
dR_dT = spline.derivative(n=1)(T_fine)

# Buigpunt & Tc Extrapolatie[cite: 20]
idx_max = np.argmax(dR_dT)
T_inflection = T_fine[idx_max]
R_inflection = R_fit[idx_max]
m_max = dR_dT[idx_max]

Tc_extrapolated = T_inflection - (R_inflection / m_max)

print(f"Buigpunt temperatuur: {T_inflection:.2f} K")
print(f"Gextrapoleerde Tc:    {Tc_extrapolated:.2f} K")

FileNotFoundError: [Errno 2] No such file or directory: 'DMMdata_pt1000.csv'

In [4]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from scipy.interpolate import UnivariateSpline

# 1. Pt1000 Kalibratie (vul jouw gemeten DMM waarden in)
R_pt1000_77K = 201.7    # Gemeten Pt1000 weerstand bij 77 K (Ohm)
R_pt1000_294K = 1088 # Gemeten Pt1000 weerstand bij 297 K (Ohm)

a_cal = (294.0 - 77.0) / (R_pt1000_294K - R_pt1000_77K)
b_cal = 77.0 - a_cal * R_pt1000_77K

def pt1000_to_kelvin(R_sensor):
    return a_cal * R_sensor + b_cal

# 2. Voorbeelddata laden/verwerken
# R_pt1000_raw: gemeten weerstand van Pt1000 op DMM (2-wire)
# V_pos_raw: gemeten spanning bij +0.1 A op SMU
# V_neg_raw: gemeten spanning bij -0.1 A op SMU

# Bereken Temperatuur
T_data = pt1000_to_kelvin(R_pt1000_raw)

# Bereken Gecorrigeerde BSCCO Weerstand (Offset Compensatie)
# R = (V_pos - V_neg) / (I_pos - I_neg)
I_pos = 0.1
I_neg = -0.1
R_bscco_corrected = (V_pos_raw - V_neg_raw) / (I_pos - I_neg)

# 3. Sorteren op stijgende temperatuur
sort_idx = np.argsort(T_data)
T_clean = T_data[sort_idx]
R_clean = R_bscco_corrected[sort_idx]

# 4. Ruisfiltering & Spline Fit voor Tc bepaling
window_len = 15 if len(R_clean) >= 15 else (len(R_clean) if len(R_clean) % 2 != 0 else len(R_clean) - 1)
R_smooth = savgol_filter(R_clean, window_length=window_len, polyorder=2)

spline = UnivariateSpline(T_clean, R_smooth, s=1e-6)
T_fine = np.linspace(T_clean.min(), T_clean.max(), 3000)
R_fit = spline(T_fine)
dR_dT = spline.derivative(n=1)(T_fine)

# Buigpunt & Tc Extrapolatie
idx_max = np.argmax(dR_dT)
T_inflection = T_fine[idx_max]
R_inflection = R_fit[idx_max]
m_max = dR_dT[idx_max]

Tc_extrapolated = T_inflection - (R_inflection / m_max)

print(f"Buigpunt temperatuur: {T_inflection:.2f} K")
print(f"Gextrapoleerde Tc:    {Tc_extrapolated:.2f} K")

NameError: name 'R_pt1000_raw' is not defined